## BRICK-FM: Running with Custom FaIR Forcing

This notebook shows how to run MimiBRICK-FM with an arbitrary external climate
forcing — for example, a single FaIR ensemble draw or a pulse trajectory — rather
than the packaged SSP scenario means.

Three patterns are covered:

1. **Packaged scenario mean** — `create_brick_fair()` with a named SSP. Simplest;
   useful for a quick deterministic run under a standard scenario.
2. **Single custom trajectory** — `get_model` + `set_external_forcing!` +
   `update_brick_mengel_params!`. Use this for a single FaIR draw, a pulse
   trajectory, or any non-standard forcing.
3. **Ensemble loop** — iterating pattern 2 over a posterior subsample to produce
   a full probabilistic projection.

All patterns use the BRICK-FM defaults: Mengel glacier emulator
(`glacier_model=:mengel`) and deterministic land-water storage (`lws=:central`).

### Step 1. Environment Setup

In [ ]:
using Pkg
Pkg.activate(".")
Pkg.instantiate()

using MimiBRICK
using CSVFiles, DataFrames

### Pattern 1: Packaged SSP scenario mean

`create_brick_fair` is a convenience wrapper that loads one of the six FaIR v2.2
mean trajectories shipped with the package (ssp119, ssp126, ssp245, ssp370,
ssp460, ssp585). It calls `get_model` + `set_external_forcing!` internally.

This runs the model at the **posterior mean** parameter values (the package
defaults); it does not apply a posterior subsample. Useful for a quick
deterministic check.

In [ ]:
m = create_brick_fair(ssprcp_scenario="ssp245", end_year=2100,
                      glacier_model=:mengel, lws=:central)
run(m)

# Total GMSL (meters, relative to 1850–1900)
gmsl = m[:global_sea_level, :sea_level_rise]
println("SSP2-4.5 GMSL at 2100 (posterior mean params): ",
        round(gmsl[end] * 100, digits=1), " cm")

### Pattern 2: Single custom FaIR trajectory

For any forcing that is not one of the packaged SSP means — a single RFF-SP draw,
a pulse experiment, an observationally-constrained historical run — use
`get_model` + `set_external_forcing!` directly.

The two required inputs are:
- `gmst`: annual global mean surface temperature anomaly relative to 1850–1900 (°C),
  length equal to `end_year - start_year + 1`
- `ohc`: cumulative ocean heat content stock (units: 1×10²² J), same length

Apply one row of the Mengel posterior with `update_brick_mengel_params!` before
running. The example below loads the shipped posterior subsample and applies the
first row.

In [ ]:
# --- Load your FaIR forcing (replace with your own arrays) -------------------
# Here we re-use the packaged SSP2-4.5 mean as a stand-in for a custom trajectory.
obs_dir   = joinpath(pkgdir(MimiBRICK), "data", "observations")
gmst_df   = DataFrame(load(joinpath(obs_dir, "fair_mean_gmst_ssp245.csv")))
ohc_df    = DataFrame(load(joinpath(obs_dir, "fair_mean_ohc_ssp245.csv")))

START_YEAR, END_YEAR = 1850, 2100
years = START_YEAR:END_YEAR

gmst = [gmst_df[gmst_df.year .== y, :gmst_C][1]    for y in years]
ohc  = [ohc_df[ ohc_df.year  .== y, :ohc_1e22J][1] for y in years]

# --- Load one row of the Mengel posterior ------------------------------------
post_path = joinpath(pkgdir(MimiBRICK), "data", "MimiBRICK",
                     "parameters_subsample_brick_mengel.csv")
posterior = DataFrame(load(post_path))
prow = posterior[1, :]   # first posterior draw; loop over rows for an ensemble

# --- Build model, inject forcing, apply parameters, run ---------------------
m = get_model(start_year=START_YEAR, end_year=END_YEAR,
              glacier_model=:mengel, lws=:central)
set_external_forcing!(m, gmst, ohc)
update_brick_mengel_params!(m, prow)
run(m)

gmsl = m[:global_sea_level, :sea_level_rise]
println("GMSL at 2100 (posterior draw 1): ", round(gmsl[end] * 100, digits=1), " cm")

### Pattern 3: Ensemble loop over posterior draws

For a probabilistic projection, iterate over all posterior rows. Each row gets
its own BRICK run; the forcing arrays can vary per draw (e.g. paired FaIR
ensemble members) or be shared (e.g. one FaIR scenario mean applied to all
posterior draws).

Note: `get_model` is called **once** outside the loop — Mimi model construction
is the expensive step. `set_external_forcing!` and `update_brick_mengel_params!`
mutate parameters in place, so the same model object can be reused.

In [ ]:
N = nrow(posterior)   # number of posterior draws
gmsl_ensemble = zeros(length(years), N)

# Build once; reuse inside the loop
m = get_model(start_year=START_YEAR, end_year=END_YEAR,
              glacier_model=:mengel, lws=:central)

for i in 1:N
    # If forcing varies per draw (e.g. paired FaIR draws), reload gmst/ohc here.
    # For a shared scenario mean, the same gmst/ohc arrays apply to every draw.
    set_external_forcing!(m, gmst, ohc)
    update_brick_mengel_params!(m, posterior[i, :])
    run(m)
    gmsl_ensemble[:, i] = m[:global_sea_level, :sea_level_rise]
end

# Posterior percentiles at 2100 (convert m → cm)
gmsl_2100_cm = gmsl_ensemble[end, :] .* 100
println("GMSL at 2100 — p5/p50/p95 (cm): ",
        round(quantile(gmsl_2100_cm, 0.05), digits=1), " / ",
        round(quantile(gmsl_2100_cm, 0.50), digits=1), " / ",
        round(quantile(gmsl_2100_cm, 0.95), digits=1))

### Notes

**Forcing units:**
- `gmst` in °C, anomaly relative to 1850–1900 (matches FaIR v2.2 output directly).
- `ohc` in 1×10²² J cumulative stock (multiply FaIR's ZJ output by 0.1 to convert).

**Posterior file:** `data/MimiBRICK/parameters_subsample_brick_mengel.csv` ships
with the package. If you have re-run the calibration, point `post_path` at your
own output from `calibration/postprocess_mcmc_mengel.jl`.

**`update_brick_mengel_params!` vs `update_brick_params!`:** use the Mengel
variant for BRICK-FM models. The original `update_brick_params!` expects the
Wong et al. (2022) posterior column set (WRB glaciers, full AIS geometry free)
and will error on a Mengel posterior row.

**Retrieving component outputs:**
```julia
m[:global_sea_level,    :sea_level_rise]   # total GMSL (m)
m[:antarctic_icesheet,  :ais_sea_level_contribution]   # AIS (m)
m[:greenland_icesheet,  :greenland_sea_level]          # GIS (m)
m[:glaciers_small_icecaps, :gic_sea_level]             # GSIC (m)
m[:thermal_expansion,   :te_sea_level]                 # TE  (m)
m[:landwater_storage,   :lws_sea_level]                # LWS (m)
```